# Notebook 05 — Conversational Harmonization

Loads the semantic output and adds conversational-level annotation:
- `previous_focus` / `previous_intent` — prior turn context
- `focus_shift` / `intent_shift` — topic/intent transition detection
- `context_dependent` — whether the utterance requires prior context
- `is_ambiguous` / `ambiguity_type` — ambiguity detection
- `expected_action` — what should happen next
- `target_response` — the actual next response when available

All processing is rule-based and deterministic.

In [1]:
import pandas as pd
import numpy as np
import re
import json
import warnings
from pathlib import Path
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
tqdm.pandas()

SEMANTIC_DIR = Path('../data/processed/semantic')
CONV_DIR = Path('../data/processed/conversational')
CONV_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_parquet(SEMANTIC_DIR / 'harmonized_semantic.parquet')
print('Loaded semantic dataset:', df.shape)

# Ensure correct sort order before any per-dialogue processing
df = df.sort_values(['dialogue_id', 'turn_id']).reset_index(drop=True)
display(df.head(6))

Loaded semantic dataset: (334033, 15)


,dialogue_id,turn_id,speaker,utterance,focus_raw,focus_normalized,primary_intent,dialogue_act,medical_entities,source_dataset,original_id,source_label_raw,dialogue_origin,annotation_source,annotation_confidence
0,healthchat_000000,0,user,NaN,Mental Health,mental_health,symptom_inquiry,question,[],HealthChat,000573958699464e9de6493b5e182fab,"{""specialty"": ""Mental Health"", ""specialty_code...",original,source_metadata,0.733
1,healthchat_000000,1,assistant,NaN,Mental Health,mental_health,other,answer,[],HealthChat,000573958699464e9de6493b5e182fab,"{""specialty"": ""Mental Health"", ""specialty_code...",original,source_metadata,0.767
2,healthchat_000000,2,user,NaN,Mental Health,mental_health,diagnosis_inquiry,question,[],HealthChat,000573958699464e9de6493b5e182fab,"{""specialty"": ""Mental Health"", ""specialty_code...",original,source_metadata,0.733
3,healthchat_000000,3,assistant,NaN,Mental Health,mental_health,other,answer,[],HealthChat,000573958699464e9de6493b5e182fab,"{""specialty"": ""Mental Health"", ""specialty_code...",original,source_metadata,0.767
4,healthchat_000000,4,user,NaN,Mental Health,mental_health,risk_factors,question,[],HealthChat,000573958699464e9de6493b5e182fab,"{""specialty"": ""Mental Health"", ""specialty_code...",original,source_metadata,0.733
5,healthchat_000001,0,user,NaN,Rheumatology,rheumatology,symptom_inquiry,question,[],HealthChat,000ac740005818a54be24a9d8ffb8e49,"{""specialty"": ""Rheumatology"", ""specialty_code""...",original,source_metadata,0.733


## 1. Previous Focus and Previous Intent

For each turn, look up the values from the immediately preceding turn in the same dialogue.
First turn of every dialogue gets `None`.

In [2]:
# ─── Build previous-turn lookup using groupby shift ────────────────────────
# Sort is already done above
df['previous_focus']  = df.groupby('dialogue_id')['focus_normalized'].shift(1)
df['previous_intent'] = df.groupby('dialogue_id')['primary_intent'].shift(1)

# shift returns NaN for first turn — that is correct
# Verify: first turn of each dialogue has NaN
first_turns = df[df['turn_id'] == 0]
assert first_turns['previous_focus'].isna().all(), 'First turn previous_focus should be None'
assert first_turns['previous_intent'].isna().all(), 'First turn previous_intent should be None'

print('previous_focus and previous_intent computed.')
print('First turn previous_focus null:', first_turns['previous_focus'].isna().all())
print('First turn previous_intent null:', first_turns['previous_intent'].isna().all())
print()
print('Non-null previous_focus:', df['previous_focus'].notna().sum())
print('Non-null previous_intent:', df['previous_intent'].notna().sum())

previous_focus and previous_intent computed.
First turn previous_focus null: True
First turn previous_intent null: True

Non-null previous_focus: 130483
Non-null previous_intent: 172434


## 2. Focus Shift

A focus shift occurs when the medical topic genuinely changes between turns.

**Logic:**
- First turn: `False` (no prior context)
- If either focus is `None`: `False` (cannot determine)
- If normalized focuses are the same string: `False`
- If focuses differ but are semantically related (same disease group): `False`
- Otherwise: `True`

**Semantic grouping** handles cases like `diabetes` vs `diabetes_type2` — same broad topic.

In [3]:
# ─── Focus grouping for shift detection ───────────────────────────────────
# Maps a normalized focus to its broad category group.
# Focuses in the same group do NOT constitute a shift.
FOCUS_GROUP = {
    # Diabetes group
    'diabetes':          'diabetes_group',
    'diabetes_type1':    'diabetes_group',
    'diabetes_type2':    'diabetes_group',
    # Cardiac group
    'heart_disease':           'cardiac_group',
    'myocardial_infarction':   'cardiac_group',
    'coronary_artery_disease': 'cardiac_group',
    'hypertension':            'cardiac_group',
    # Cancer group
    'cancer':             'cancer_group',
    'breast_cancer':      'cancer_group',
    'lung_cancer':        'cancer_group',
    'prostate_cancer':    'cancer_group',
    'colorectal_cancer':  'cancer_group',
    # Mental health group
    'mental_health':  'mental_health_group',
    'depression':     'mental_health_group',
    'anxiety':        'mental_health_group',
    # Respiratory group
    'asthma':  'respiratory_group',
    'copd':    'respiratory_group',
    # Neurological group
    'stroke':             'neuro_group',
    'alzheimers_disease': 'neuro_group',
    'parkinsons_disease': 'neuro_group',
    'migraine':           'neuro_group',
    'headache':           'neuro_group',
    # Arthritis group
    'arthritis':           'arthritis_group',
    'rheumatoid_arthritis': 'arthritis_group',
    'osteoarthritis':      'arthritis_group',
    # Thyroid group
    'thyroid_disorder':  'thyroid_group',
    'hypothyroidism':    'thyroid_group',
    'hyperthyroidism':   'thyroid_group',
    # GI group
    'ibs':   'gi_group',
    'gerd':  'gi_group',
}

def get_focus_group(focus):
    """Return the broad group for a focus, or the focus itself if not grouped."""
    if focus is None or (isinstance(focus, float) and pd.isna(focus)):
        return None
    return FOCUS_GROUP.get(str(focus), str(focus))

def is_focus_shift(current_focus, previous_focus, turn_id):
    """Determine if a focus shift occurred."""
    if turn_id == 0:
        return False
    if current_focus is None or pd.isna(current_focus):
        return False
    if previous_focus is None or pd.isna(previous_focus):
        return False
    current_group = get_focus_group(current_focus)
    previous_group = get_focus_group(previous_focus)
    return current_group != previous_group

df['focus_shift'] = df.apply(
    lambda r: is_focus_shift(r['focus_normalized'], r['previous_focus'], r['turn_id']),
    axis=1
)

# Verify first turns are False
assert not df[df['turn_id'] == 0]['focus_shift'].any()

print('focus_shift distribution:', df['focus_shift'].value_counts().to_dict())
print('Focus shift rate:', f"{df['focus_shift'].mean():.3f}")

# Show examples of detected focus shifts
shifts = df[df['focus_shift'] == True][['dialogue_id', 'turn_id', 'previous_focus', 'focus_normalized', 'utterance']].head(5)
print('\nExample focus shifts:')
display(shifts)

focus_shift distribution: {False: 305182, True: 28851}
Focus shift rate: 0.086

Example focus shifts:


,dialogue_id,turn_id,previous_focus,focus_normalized,utterance
76882,meddialog_0000001,1,skin_condition,infection,Hi... Thank you for consulting in Chat Doctor....
76884,meddialog_0000002,1,pain,pregnancy,"Hello, and I hope I can help you today.First, ..."
76888,meddialog_0000004,1,fever,pneumonia,Thank you for using Chat Doctor. I would sugge...
76896,meddialog_0000008,1,influenza,infection,"Hi, If the symptoms persist that long this sug..."
76904,meddialog_0000012,1,infection,skin_condition,Thanks for your question on Chat Doctor. I can...


## 3. Intent Shift

In [4]:
def is_intent_shift(current_intent, previous_intent, turn_id):
    """Determine if an intent shift occurred."""
    if turn_id == 0:
        return False
    if current_intent is None or pd.isna(current_intent):
        return False
    if previous_intent is None or pd.isna(previous_intent):
        return False
    return str(current_intent) != str(previous_intent)

df['intent_shift'] = df.apply(
    lambda r: is_intent_shift(r['primary_intent'], r['previous_intent'], r['turn_id']),
    axis=1
)

assert not df[df['turn_id'] == 0]['intent_shift'].any()

print('intent_shift distribution:', df['intent_shift'].value_counts().to_dict())
print('Intent shift rate:', f"{df['intent_shift'].mean():.3f}")

intent_shift distribution: {True: 168314, False: 165719}
Intent shift rate: 0.504


## 4. Context Dependence

A turn is context-dependent when its meaning or referent cannot be resolved without prior turns.

**Heuristics:**
1. Starts with a pronoun referring to a prior entity: `it`, `this`, `that`, `they`, `he`, `she`, `these`, `those`
2. Uses `What about`, `How about`, `And the`, `Is it`, `Can I`, etc.
3. Very short question (<= 6 words) containing a pronoun
4. References `the other`, `the same`, `the medication`, `the condition` etc. (definite article + medical noun)
5. First turn: always `False`

In [5]:
RE_PRONOUN_START = re.compile(r'^\s*(it|this|that|they|he|she|these|those|its|their)\b', re.I)
RE_CONTEXT_PHRASE = re.compile(
    r'\b(what about|how about|and the|and what|is it|is this|can i|can it|'
    r'does it|do they|how long|how much|how many|will it|would it|'
    r'the other|the same|the medication|the treatment|the condition|'
    r'the symptom|the disease|the drug|the test|mentioned earlier|'
    r'as mentioned|from before|you mentioned|you said)\b',
    re.I
)
RE_DEMONSTRATIVE = re.compile(r'\b(this condition|this disease|this medication|this treatment|'
                               r'that medication|that treatment|those symptoms|these symptoms)\b', re.I)

def is_context_dependent(utterance, turn_id, speaker):
    """Determine if an utterance is context-dependent."""
    if turn_id == 0:
        return False
    if utterance is None or (isinstance(utterance, float) and pd.isna(utterance)):
        return False  # Can't tell without text
    text = str(utterance).strip()
    # Short question with pronoun
    words = text.split()
    if len(words) <= 6 and RE_PRONOUN_START.match(text):
        return True
    if RE_PRONOUN_START.match(text):
        return True
    if RE_CONTEXT_PHRASE.search(text):
        return True
    if RE_DEMONSTRATIVE.search(text):
        return True
    return False

df['context_dependent'] = df.apply(
    lambda r: is_context_dependent(r['utterance'], r['turn_id'], r['speaker']),
    axis=1
)

assert not df[df['turn_id'] == 0]['context_dependent'].any()

print('context_dependent distribution:', df['context_dependent'].value_counts().to_dict())
print('Context dependence rate:', f"{df['context_dependent'].mean():.3f}")

# Show examples
ctx_examples = df[df['context_dependent'] == True][['dialogue_id','turn_id','speaker','utterance']].head(8)
print()
print('Example context-dependent turns:')
display(ctx_examples)

context_dependent distribution: {False: 297569, True: 36464}
Context dependence rate: 0.109

Example context-dependent turns:


,dialogue_id,turn_id,speaker,utterance
76880,meddialog_0000000,1,assistant,"Hi, Thank you for posting your query. The most..."
76884,meddialog_0000002,1,assistant,"Hello, and I hope I can help you today.First, ..."
76886,meddialog_0000003,1,assistant,HI. You have two different problems. The lump ...
76910,meddialog_0000015,1,assistant,HiT hanks for choosing Chat Doctor for your qu...
76920,meddialog_0000020,1,assistant,"Hi. Thanks for your query, read and understood..."
76928,meddialog_0000024,1,assistant,HelloThanks for posting at Chat Doctor. You ha...
76946,meddialog_0000033,1,assistant,"Hallow Dear, Since you are no more on the birt..."
76970,meddialog_0000045,1,assistant,"Hello, I can understand your concern. Usually,..."


## 5. Ambiguity Detection

**Ambiguity types:**
```
referential, lexical, temporal, clinical, intent, contextual, scope, none
```

**Rules:**
- `referential`: Pronoun reference to undefined entity (it, this, that without prior context)
- `lexical`: Multiple meanings possible (e.g. `cold`, `elevated`)
- `temporal`: Unclear time reference (e.g. `recently`, `lately`, `sometimes`)
- `clinical`: Clinically underspecified (e.g. `Is it serious?`, `Is it dangerous?`)
- `intent`: Ambiguous intent (e.g. asking for information vs. seeking reassurance)
- `contextual`: Requires prior context that is not provided
- `scope`: Unclear scope/referent of question
- `none`: No ambiguity detected

In [6]:
VALID_AMBIGUITY_TYPES = {'referential', 'lexical', 'temporal', 'clinical', 'intent', 'contextual', 'scope', 'none'}

RE_REFERENTIAL = re.compile(
    r'^\s*(is it|is this|is that|are they|can it|does it|will it|would it|'
    r'what is it|what does it|how does it|what about it|is it serious|'
    r'is it dangerous|is it contagious|is it curable|is it hereditary|'
    r'can i take it|should i take it|how long does it)\b',
    re.I
)
RE_LEXICAL = re.compile(
    r'\b(cold|elevated|high|low|normal|positive|negative|mild|moderate|'
    r'chronic|acute|discharge|mass|growth)\b',
    re.I
)
RE_TEMPORAL = re.compile(r'\b(recently|lately|sometimes|occasionally|often|for a while|'
                          r'last week|last month|last year|few days|a while ago|some time)\b', re.I)
RE_CLINICAL = re.compile(r'\b(is it serious|is it dangerous|is it bad|could it be|'
                          r'am i at risk|should i be worried|is that normal|is this normal|'
                          r'do i need to see|should i see a doctor|is it life.threatening)\b', re.I)
RE_SCOPE = re.compile(r'\b(what else|anything else|what other|are there other|'
                       r'any other|other options|other treatments|other causes)\b', re.I)

def detect_ambiguity(utterance, turn_id, speaker, context_dependent):
    """
    Detect ambiguity in an utterance.
    Returns (is_ambiguous: bool, ambiguity_type: str).
    Only marks user turns — assistant answers are not marked ambiguous.
    """
    if speaker == 'assistant':
        return False, 'none'
    if utterance is None or (isinstance(utterance, float) and pd.isna(utterance)):
        return False, 'none'

    text = str(utterance).strip()

    # Referential ambiguity — pronoun with no clear referent
    if RE_REFERENTIAL.match(text):
        return True, 'referential'

    # Clinical ambiguity — question about severity/risk without context
    if RE_CLINICAL.search(text):
        return True, 'clinical'

    # Contextual — context-dependent + ambiguous referent
    if context_dependent and RE_PRONOUN_START.match(text):
        return True, 'contextual'

    # Scope ambiguity
    if RE_SCOPE.search(text):
        return True, 'scope'

    # Temporal ambiguity — only if short + temporal word
    words = text.split()
    if len(words) <= 15 and RE_TEMPORAL.search(text) and '?' not in text:
        return True, 'temporal'

    # Lexical ambiguity — only for short utterances where meaning is unclear
    if len(words) <= 8 and RE_LEXICAL.search(text):
        return True, 'lexical'

    return False, 'none'

ambiguity_results = df.apply(
    lambda r: detect_ambiguity(r['utterance'], r['turn_id'], r['speaker'], r['context_dependent']),
    axis=1
)
df['is_ambiguous'] = ambiguity_results.apply(lambda x: x[0])
df['ambiguity_type'] = ambiguity_results.apply(lambda x: x[1])

print('is_ambiguous distribution:', df['is_ambiguous'].value_counts().to_dict())
print('ambiguity_type distribution:')
print(df['ambiguity_type'].value_counts())
print(f'Ambiguity rate: {df["is_ambiguous"].mean():.3f}')

# Validate
invalid_amb_type = df[~df['ambiguity_type'].isin(VALID_AMBIGUITY_TYPES)]
print(f'Invalid ambiguity_type values: {len(invalid_amb_type)}')
assert len(invalid_amb_type) == 0

# Show examples
print('\nExample ambiguous turns:')
display(df[df['is_ambiguous'] == True][['dialogue_id','turn_id','speaker','utterance','ambiguity_type']].head(8))

is_ambiguous distribution: {False: 326640, True: 7393}
ambiguity_type distribution:
ambiguity_type
none           326640
clinical         4388
scope            2690
referential       166
lexical           148
temporal            1
Name: count, dtype: int64
Ambiguity rate: 0.022
Invalid ambiguity_type values: 0

Example ambiguous turns:


,dialogue_id,turn_id,speaker,utterance,ambiguity_type
76911,meddialog_0000016,0,user,I had a alt reading about 2+ months ago of 43 ...,clinical
76913,meddialog_0000017,0,user,i had surgery done 5 months ago for scar tissu...,clinical
76931,meddialog_0000026,0,user,I too fell and hurt my left side of my chest. ...,clinical
76991,meddialog_0000056,0,user,"Hi, i ve been on the Noriday pill for 10 month...",clinical
77011,meddialog_0000066,0,user,I have noticed that I ve been getting these so...,clinical
77017,meddialog_0000069,0,user,"Hi Sir, My mother is 68yeasr old, and from ver...",scope
77101,meddialog_0000111,0,user,"HELLO i AM 33 YRS OF AGE, IM 510 AND WEIGH 225...",scope
77119,meddialog_0000120,0,user,Hi my left bottom wisdom tooth is coming throu...,clinical


## 6. Expected Action

In [7]:
VALID_EXPECTED_ACTIONS = {
    'answer', 'ask_clarification', 'provide_information', 'provide_safety_guidance',
    'provide_medical_context', 'request_missing_information',
    'acknowledge', 'redirect', 'other'
}

def infer_expected_action(speaker, intent, dialogue_act, is_ambiguous, ambiguity_type, utterance):
    """Infer the expected response action for this turn."""
    if speaker == 'assistant':
        # Assistant turns don't need an 'expected action' — they ARE the action
        return 'other'

    # Ambiguous turns where context is insufficient
    if is_ambiguous and ambiguity_type in ('referential', 'contextual', 'clinical'):
        return 'ask_clarification'

    # Emergency/urgent — safety guidance first
    if intent == 'emergency_or_urgent':
        return 'provide_safety_guidance'

    # Direct intent mappings
    intent_action_map = {
        'information_seeking':  'provide_information',
        'symptom_inquiry':      'provide_medical_context',
        'diagnosis_inquiry':    'provide_medical_context',
        'treatment_inquiry':    'provide_information',
        'medication_inquiry':   'provide_information',
        'test_or_diagnosis':    'provide_information',
        'prevention':           'provide_information',
        'risk_factors':         'provide_information',
        'cause_or_mechanism':   'provide_information',
        'prognosis':            'provide_medical_context',
        'follow_up':            'answer',
        'clarification':        'answer',
        'other':                'answer',
    }

    if intent in intent_action_map:
        return intent_action_map[intent]

    # Fallback
    if dialogue_act == 'question':
        return 'answer'
    return 'other'

df['expected_action'] = df.apply(
    lambda r: infer_expected_action(
        r['speaker'], r['primary_intent'], r['dialogue_act'],
        r['is_ambiguous'], r['ambiguity_type'], r['utterance']
    ),
    axis=1
)

print('expected_action distribution:')
print(df['expected_action'].value_counts())

invalid_action = df[~df['expected_action'].isin(VALID_EXPECTED_ACTIONS)]
print(f'Invalid expected_action values: {len(invalid_action)}')
assert len(invalid_action) == 0

expected_action distribution:
expected_action
other                      154987
provide_information        126101
provide_medical_context     39877
provide_safety_guidance      4671
ask_clarification            4554
answer                       3843
Name: count, dtype: int64
Invalid expected_action values: 0


## 7. Target Response

**Policy:**
- For **MedQuAD** (constructed 2-turn): `target_response` for the user turn (turn 0) = the assistant answer (turn 1). Assistant turn gets `None`.
- For **MedDialog** (constructed 2-turn): same as MedQuAD.
- For **HealthChat** (original, multi-turn, no text): `None` for all turns since no utterance text is available.

No responses are fabricated.

In [8]:
# ─── Build next-turn utterance lookup ─────────────────────────────────────
# For each user turn (turn_id=0 in 2-turn dialogues), target = the assistant utterance
# Use groupby shift(-1) to get the next turn's utterance within each dialogue

df['target_response'] = df.groupby('dialogue_id')['utterance'].shift(-1)

# For assistant turns, target_response should be None (they don't need a target)
df.loc[df['speaker'] == 'assistant', 'target_response'] = None

# For HealthChat (no text), target_response is already None since utterance is None
# Verify HealthChat stays None
hc_targets = df[df['source_dataset'] == 'HealthChat']['target_response'].notna().sum()
print(f'HealthChat non-null target_responses (should be 0 or very few): {hc_targets}')

# Stats
has_target = df['target_response'].notna().sum()
print(f'Turns with target_response: {has_target:,} / {len(df):,}')
print()
print('Target response by dataset:')
print(df.groupby('source_dataset')['target_response'].apply(lambda x: x.notna().sum()))

HealthChat non-null target_responses (should be 0 or very few): 0
Turns with target_response: 128,572 / 334,033

Target response by dataset:
source_dataset
HealthChat         0
MedDialog     112165
MedQuAD        16407
Name: target_response, dtype: int64


## 8. Assemble Final Conversational DataFrame and Save

In [9]:
CONV_COLS = [
    'dialogue_id', 'turn_id', 'speaker', 'utterance',
    'focus_raw', 'focus_normalized', 'primary_intent', 'dialogue_act',
    'medical_entities',
    'previous_focus', 'previous_intent',
    'focus_shift', 'intent_shift',
    'context_dependent', 'is_ambiguous', 'ambiguity_type',
    'expected_action', 'target_response',
    'source_dataset', 'original_id', 'source_label_raw', 'dialogue_origin',
    'annotation_source', 'annotation_confidence'
]

conv_df = df[CONV_COLS].copy()

# Final sort
conv_df = conv_df.sort_values(['dialogue_id', 'turn_id']).reset_index(drop=True)

print('Conversational dataset shape:', conv_df.shape)
display(conv_df.head(8))

Conversational dataset shape: (334033, 24)


,dialogue_id,turn_id,speaker,utterance,focus_raw,focus_normalized,primary_intent,dialogue_act,medical_entities,previous_focus,...,is_ambiguous,ambiguity_type,expected_action,target_response,source_dataset,original_id,source_label_raw,dialogue_origin,annotation_source,annotation_confidence
0,healthchat_000000,0,user,NaN,Mental Health,mental_health,symptom_inquiry,question,[],NaN,...,False,none,provide_medical_context,NaN,HealthChat,000573958699464e9de6493b5e182fab,"{""specialty"": ""Mental Health"", ""specialty_code...",original,source_metadata,0.733
1,healthchat_000000,1,assistant,NaN,Mental Health,mental_health,other,answer,[],mental_health,...,False,none,other,NaN,HealthChat,000573958699464e9de6493b5e182fab,"{""specialty"": ""Mental Health"", ""specialty_code...",original,source_metadata,0.767
2,healthchat_000000,2,user,NaN,Mental Health,mental_health,diagnosis_inquiry,question,[],mental_health,...,False,none,provide_medical_context,NaN,HealthChat,000573958699464e9de6493b5e182fab,"{""specialty"": ""Mental Health"", ""specialty_code...",original,source_metadata,0.733
3,healthchat_000000,3,assistant,NaN,Mental Health,mental_health,other,answer,[],mental_health,...,False,none,other,NaN,HealthChat,000573958699464e9de6493b5e182fab,"{""specialty"": ""Mental Health"", ""specialty_code...",original,source_metadata,0.767
4,healthchat_000000,4,user,NaN,Mental Health,mental_health,risk_factors,question,[],mental_health,...,False,none,provide_information,NaN,HealthChat,000573958699464e9de6493b5e182fab,"{""specialty"": ""Mental Health"", ""specialty_code...",original,source_metadata,0.733
5,healthchat_000001,0,user,NaN,Rheumatology,rheumatology,symptom_inquiry,question,[],NaN,...,False,none,provide_medical_context,NaN,HealthChat,000ac740005818a54be24a9d8ffb8e49,"{""specialty"": ""Rheumatology"", ""specialty_code""...",original,source_metadata,0.733
6,healthchat_000002,0,user,NaN,Cardiology,cardiology,other,question,[],NaN,...,False,none,answer,NaN,HealthChat,00110769d51fdd16b174e05e6407c50a,"{""specialty"": ""Cardiology"", ""specialty_code"": ...",original,source_metadata,0.567
7,healthchat_000003,0,user,NaN,Other,general_medicine,information_seeking,question,[],NaN,...,False,none,provide_information,NaN,HealthChat,0013a16e8e9064969120c69ec83c51b5,"{""specialty"": ""Other"", ""specialty_code"": 21, ""...",original,source_metadata,0.600


In [10]:
out_path = CONV_DIR / 'harmonized_conversational.parquet'
conv_df.to_parquet(out_path, index=False)
print(f'Saved: {out_path}')
print(f'Size:  {out_path.stat().st_size / 1024 / 1024:.1f} MB')

print()
print('=== CONVERSATIONAL HARMONIZATION SUMMARY ===')
print(f'Total turns:        {len(conv_df):,}')
print(f'Total dialogues:    {conv_df["dialogue_id"].nunique():,}')
print(f'Context-dependent:  {conv_df["context_dependent"].sum():,} ({100*conv_df["context_dependent"].mean():.1f}%)')
print(f'Ambiguous turns:    {conv_df["is_ambiguous"].sum():,} ({100*conv_df["is_ambiguous"].mean():.1f}%)')
print(f'Focus shifts:       {conv_df["focus_shift"].sum():,} ({100*conv_df["focus_shift"].mean():.1f}%)')
print(f'Intent shifts:      {conv_df["intent_shift"].sum():,} ({100*conv_df["intent_shift"].mean():.1f}%)')
print()
print('Ambiguity type distribution:')
print(conv_df['ambiguity_type'].value_counts())

Saved: ..\data\processed\conversational\harmonized_conversational.parquet
Size:  132.1 MB

=== CONVERSATIONAL HARMONIZATION SUMMARY ===
Total turns:        334,033
Total dialogues:    161,599
Context-dependent:  36,464 (10.9%)
Ambiguous turns:    7,393 (2.2%)
Focus shifts:       28,851 (8.6%)
Intent shifts:      168,314 (50.4%)

Ambiguity type distribution:
ambiguity_type
none           326640
clinical         4388
scope            2690
referential       166
lexical           148
temporal            1
Name: count, dtype: int64
